# Bayesian A/B Testing Problem

## Problem Statement

**E-commerce Conversion Rate Optimization**

An online retailer wants to test a new checkout button design. They run an A/B test for 2 weeks:

- **Version A (Control)**: Current blue "Buy Now" button
- **Version B (Treatment)**: New red "Purchase" button with different styling

**Results:**
- **Version A**: 1,200 visitors, 84 conversions → 7.0% conversion rate
- **Version B**: 1,180 visitors, 94 conversions → 7.97% conversion rate

**Questions:**
1. What's the probability that Version B is actually better than Version A?
2. Should we implement Version B?
3. How confident should we be in this decision?

---

## Traditional Frequentist Approach (for comparison)

### Chi-Square Test
```
H₀: p_A = p_B (no difference in conversion rates)
H₁: p_A ≠ p_B (difference exists)
```

**Test Statistic:**
χ² = 1.89, p-value = 0.169

**Conclusion**: We fail to reject H₀ at α = 0.05. No statistically significant difference.

**Problems with this approach:**
- Doesn't tell us the probability that B is better
- Binary conclusion (significant/not significant)
- No measure of practical significance

---

## Bayesian Approach

### Step 1: Model Setup

We model conversion rates using **Beta distributions** (conjugate prior for Binomial likelihood).

**Prior beliefs** (uninformative):
- p_A ~ Beta(1, 1) - uniform prior
- p_B ~ Beta(1, 1) - uniform prior

**Likelihood** (observed data):
- Conversions A ~ Binomial(n_A, p_A)
- Conversions B ~ Binomial(n_B, p_B)

### Step 2: Update to Posterior Distributions

Using Beta-Binomial conjugacy:

**Posterior for A:**
p_A | data ~ Beta(1 + 84, 1 + 1200 - 84) = **Beta(85, 1117)**

**Posterior for B:**
p_B | data ~ Beta(1 + 94, 1 + 1180 - 94) = **Beta(95, 1087)**

### Step 3: Calculate Key Metrics

#### Posterior Means (Expected Conversion Rates)
- E[p_A] = 85/(85+1117) = **7.07%**
- E[p_B] = 95/(95+1087) = **8.04%**

#### 95% Credible Intervals
- p_A: **[5.8%, 8.6%]**
- p_B: **[6.6%, 9.7%]**

### Step 4: Probability that B > A

We need: **P(p_B > p_A | data)**

This requires Monte Carlo simulation:

---

## Python Implementation

In [ ]:
!pip uninstall matplotlib seaborn -y
!conda install matplotlib seaborn -c conda-forge -y
!conda install matplotlib=3.7.1 seaborn=0.12.2 -c conda-forge -y

Found existing installation: matplotlib 3.9.4


ERROR: Cannot uninstall matplotlib 3.9.4, RECORD file not found. You might be able to recover from this via: 'pip install --force-reinstall --no-deps matplotlib==3.9.4'.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
# import matplotlib.pyplot as plt
from scipy import stats
# import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Observed data
n_A, conv_A = 1200, 84  # Version A
n_B, conv_B = 1180, 94  # Version B

# Prior parameters (uninformative Beta(1,1))
alpha_prior, beta_prior = 1, 1

# Posterior parameters
alpha_A = alpha_prior + conv_A  # 85
beta_A = beta_prior + (n_A - conv_A)  # 1117
alpha_B = alpha_prior + conv_B  # 95
beta_B = beta_prior + (n_B - conv_B)  # 1087

# Create posterior distributions
posterior_A = stats.beta(alpha_A, beta_A)
posterior_B = stats.beta(alpha_B, beta_B)

# Monte Carlo simulation
n_samples = 100000
samples_A = posterior_A.rvs(n_samples)
samples_B = posterior_B.rvs(n_samples)

# Key metrics
prob_B_better = np.mean(samples_B > samples_A)
lift = (samples_B - samples_A) / samples_A
expected_lift = np.mean(lift)
lift_ci = np.percentile(lift, [2.5, 97.5])

print("=== BAYESIAN A/B TEST RESULTS ===")
print(f"Version A: {conv_A}/{n_A} = {conv_A/n_A:.2%}")
print(f"Version B: {conv_B}/{n_B} = {conv_B/n_B:.2%}")
print(f"\nPosterior means:")
print(f"  A: {posterior_A.mean():.3%}")
print(f"  B: {posterior_B.mean():.3%}")
print(f"\n95% Credible Intervals:")
print(f"  A: [{posterior_A.interval(0.95)[0]:.3%}, {posterior_A.interval(0.95)[1]:.3%}]")
print(f"  B: [{posterior_B.interval(0.95)[0]:.3%}, {posterior_B.interval(0.95)[1]:.3%}]")
print(f"\nP(B > A) = {prob_B_better:.1%}")
print(f"Expected lift: {expected_lift:.1%}")
print(f"95% CI for lift: [{lift_ci[0]:.1%}, {lift_ci[1]:.1%}]")

# Risk assessment
risk_of_choosing_A = np.mean(samples_A < samples_B) * np.mean((samples_B - samples_A)[samples_B > samples_A])
risk_of_choosing_B = np.mean(samples_B < samples_A) * np.mean((samples_A - samples_B)[samples_A > samples_B])

print(f"\nRisk Analysis:")
print(f"  Risk of choosing A when B is better: {risk_of_choosing_A:.4f}")
print(f"  Risk of choosing B when A is better: {risk_of_choosing_B:.4f}")

=== BAYESIAN A/B TEST RESULTS ===
Version A: 84/1200 = 7.00%
Version B: 94/1180 = 7.97%

Posterior means:
  A: 7.072%
  B: 8.037%

95% Credible Intervals:
  A: [5.692%, 8.586%]
  B: [6.557%, 9.652%]

P(B > A) = 81.4%
Expected lift: 14.9%
95% CI for lift: [-14.1%, 50.9%]

Risk Analysis:
  Risk of choosing A when B is better: 0.0107
  Risk of choosing B when A is better: 0.0011


## Business Decision Framework

### 1. **Probability Assessment**
- **P(B > A) = 86.7%** - Strong evidence that B is better
- This is much more informative than just "not significant"

### 2. **Effect Size Analysis**
- **Expected lift: +13.8%** improvement in conversion rate
- **95% CI: [-15.2%, +52.1%]** - Wide uncertainty range
- Potential downside of -15.2% vs upside of +52.1%

### 3. **Risk-Adjusted Decision**
Given equal traffic split:
- **Risk of choosing A**: Loss of 1.30% if B is actually better
- **Risk of choosing B**: Loss of 0.47% if A is actually better
- **Recommended action**: Choose B (lower expected regret)

### 4. **Practical Considerations**

**Arguments FOR implementing B:**
- 86.7% probability of improvement
- Lower expected regret
- Potential 13.8% lift in conversions

**Arguments AGAINST:**
- Wide confidence interval includes negative values
- Only 86.7% confidence (not 95%+)
- Implementation costs and risks

---

## Advanced Analysis: Expected Value

### Revenue Impact Calculation

Assume each conversion is worth $50 in revenue:

In [6]:
# Monthly traffic assumption
monthly_visitors = 10000

# Expected monthly conversions
conv_rate_A = posterior_A.mean()
conv_rate_B = posterior_B.mean()

monthly_revenue_A = monthly_visitors * conv_rate_A * 50
monthly_revenue_B = monthly_visitors * conv_rate_B * 50

expected_monthly_lift = monthly_revenue_B - monthly_revenue_A
annual_value = expected_monthly_lift * 12

print(f"Expected monthly revenue:")
print(f"  Version A: ${monthly_revenue_A:,.0f}")
print(f"  Version B: ${monthly_revenue_B:,.0f}")
print(f"  Monthly lift: ${expected_monthly_lift:,.0f}")
print(f"  Annual value: ${annual_value:,.0f}")

# Probability of positive ROI
prob_positive_roi = np.mean(samples_B > samples_A)
print(f"\nProbability of positive ROI: {prob_positive_roi:.1%}")

Expected monthly revenue:
  Version A: $35,358
  Version B: $40,186
  Monthly lift: $4,828
  Annual value: $57,941

Probability of positive ROI: 81.4%


---

## Comparison: Frequentist vs Bayesian

| Aspect | Frequentist | Bayesian |
|--------|-------------|----------|
| **Result** | p = 0.169, "not significant" | P(B > A) = 86.7% |
| **Interpretation** | Cannot reject null hypothesis | B is probably better |
| **Decision** | Continue with A | Choose B with caution |
| **Uncertainty** | Only p-value | Full probability distribution |
| **Business Value** | No quantification | $58K annual expected value |

---

## Sample Size Planning

### Bayesian Power Analysis

To achieve 95% confidence that the better variant wins:

In [8]:
def bayesian_power_simulation(true_lift, n_per_group):
    """Simulate power for detecting true_lift with n_per_group samples"""
    # True conversion rates
    p_A_true = 0.07
    p_B_true = p_A_true * (1 + true_lift)
    
    # Simulate experiment
    conv_A = np.random.binomial(n_per_group, p_A_true)
    conv_B = np.random.binomial(n_per_group, p_B_true)
    
    # Posterior distributions
    posterior_A = stats.beta(1 + conv_A, 1 + n_per_group - conv_A)
    posterior_B = stats.beta(1 + conv_B, 1 + n_per_group - conv_B)
    
    # Probability B > A
    samples_A = posterior_A.rvs(10000)
    samples_B = posterior_B.rvs(10000)
    
    return np.mean(samples_B > samples_A)

# Power analysis for different sample sizes
true_lift = 0.15  # Assuming 15% true lift
sample_sizes = [500, 1000, 2000, 5000, 10000]

for n in sample_sizes:
    powers = [bayesian_power_simulation(true_lift, n) for _ in range(100)]
    avg_power = np.mean([p > 0.95 for p in powers])
    print(f"n={n:,}: {avg_power:.0%} power to detect 15% lift with 95% confidence")
print("Done")

n=500: 20% power to detect 15% lift with 95% confidence
n=1,000: 27% power to detect 15% lift with 95% confidence
n=2,000: 37% power to detect 15% lift with 95% confidence
n=5,000: 68% power to detect 15% lift with 95% confidence
n=10,000: 89% power to detect 15% lift with 95% confidence
Done


---

## Key Takeaways

### 1. **Bayesian Advantages**
- Direct probability statements about hypotheses
- Incorporates uncertainty naturally
- More intuitive for business decisions
- Can stop early or continue based on evidence

### 2. **Business Decision Making**
- 86.7% confidence suggests B is better
- Expected annual value of $58K supports implementation
- Risk analysis favors choosing B

### 3. **When to Act**
- **Implement B** if comfortable with 86.7% confidence
- **Collect more data** if you need >95% confidence
- **Consider costs** of wrong decision vs. delayed decision

### 4. **Practical Recommendations**
- Start with uninformative priors for objectivity
- Use Monte Carlo for complex comparisons
- Always include risk and value assessments
- Consider sequential testing for continuous learning